# 05 - Explainability
Which features push a customer toward default? Global view (coefficients) and local view (SHAP for one customer).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # so we can import from src/

import joblib
import pandas as pd
import matplotlib.pyplot as plt
from src import config
from src.data_preprocessing import load_processed_data

model = joblib.load(config.MODEL_PATH)
X_train, X_test, y_train, y_test = load_processed_data()
print("Model loaded:", type(model).__name__)

### Global view: top risk drivers (logistic regression coefficients)

In [ ]:
importance = pd.Series(model.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False)
print(importance.head(10))

importance.head(10).sort_values().plot(kind="barh", figsize=(8, 5), title="Top 10 Risk Drivers")
plt.show()

### Local view: explain one customer with SHAP

In [ ]:
import shap

sample = X_test.iloc[[0]]
proba = model.predict_proba(sample)[0][1]
print("Default Probability:", round(proba, 3))
print("Decision:", config.risk_band(proba))

explainer = shap.Explainer(model, X_train)
shap_values = explainer(sample)
shap.plots.waterfall(shap_values[0])

### Confusion matrix and headline metrics (threshold 0.5)

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             accuracy_score, precision_score, recall_score, roc_auc_score)

default_probs = model.predict_proba(X_test)[:, 1]
threshold = 0.5
y_pred = (default_probs >= threshold).astype(int)

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["No Default", "Default"]).plot(cmap="Blues")
plt.title(f"Confusion Matrix (threshold={threshold})")
plt.show()

print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, default_probs):.3f}")

### Decisions for the first 5 test customers
Uses the same thresholds as notebook 04 (from `src/config.py`).

In [ ]:
for i in range(5):
    p = default_probs[i]
    print(f"Customer {i+1}: Probability={p:.3f}, Decision={config.risk_band(p)}")